In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, RocCurveDisplay)


# LOAD & INSPECT

In [6]:
df = pd.read_csv('dataset_03_employee_attrition.csv')
print("="*70)
print("1. DATASET SHAPE:", df.shape)
print("="*70)
print(df.dtypes)
print("\nFirst rows:\n", df.head())
print("\nSummary stats:\n", df.describe())

1. DATASET SHAPE: (1000, 7)
age                   int64
monthly_income      float64
years_at_company    float64
job_satisfaction      int64
overtime_hours      float64
promotion_years     float64
target                int64
dtype: object

First rows:
    age  monthly_income  years_at_company  job_satisfaction  overtime_hours  \
0   18          554.86              2.69                 1            8.30   
1   60          104.33              5.99                 4            1.00   
2   36          205.49              0.82                 2            7.03   
3   25          372.61              0.00                 3            9.18   
4   43          228.28              8.66                 7           14.67   

   promotion_years  target  
0             1.22       1  
1             7.71       1  
2             2.80       0  
3             0.00       1  
4             5.11       0  

Summary stats:
                age  monthly_income  years_at_company  job_satisfaction  \
count  1000.00

# 2. DATA QUALITY CHECKS

In [7]:
print("\n" + "="*70)
print("2. DATA QUALITY CHECKS")
print("="*70)
print("Missing values per column:\n", df.isnull().sum())
n_dupes = df.duplicated().sum()
print(f"\nDuplicate rows: {n_dupes}")
df = df.drop_duplicates().reset_index(drop=True)
print(f"Shape after dropping duplicates: {df.shape}")
 
print("\nClass balance (target):")
print(df['target'].value_counts())
print(df['target'].value_counts(normalize=True).round(3))


2. DATA QUALITY CHECKS
Missing values per column:
 age                 0
monthly_income      0
years_at_company    0
job_satisfaction    0
overtime_hours      0
promotion_years     0
target              0
dtype: int64

Duplicate rows: 0
Shape after dropping duplicates: (1000, 7)

Class balance (target):
target
1    500
0    500
Name: count, dtype: int64
target
1    0.5
0    0.5
Name: proportion, dtype: float64


# 3. FEATURES / TARGET

In [8]:
target = 'target'
feature_cols = ['age', 'monthly_income', 'years_at_company',
                 'job_satisfaction', 'overtime_hours', 'promotion_years']
X = df[feature_cols]
y = df[target]

# 4. TRAIN/TEST SPLIT (stratified to preserve class balance -> avoids leakage)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")


Train size: 800, Test size: 200


# 5. PREPROCESSING - scale numeric features

In [10]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

 # 6. MODEL TRAINING

In [11]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)
print("\nModel configuration:")
print(model)


Model configuration:
LogisticRegression(max_iter=1000, random_state=42)


# 7. EVALUATION

In [12]:
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]
 
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)
 
print("\n" + "="*70)
print("7. EVALUATION METRICS")
print("="*70)
print(f"Accuracy : {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall   : {rec:.3f}")
print(f"F1-score : {f1:.3f}")
print(f"ROC-AUC  : {auc:.3f}")
print("\nFull classification report:\n", classification_report(y_test, y_pred))
 
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)
 
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Stayed','Left'], yticklabels=['Stayed','Left'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Employee Attrition')
plt.tight_layout()
plt.savefig('/home/claude/project/confusion_matrix.png', dpi=150)
plt.close()
 
plt.figure(figsize=(5,4))
RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title('ROC Curve - Employee Attrition')
plt.tight_layout()
plt.savefig('/home/claude/project/roc_curve.png', dpi=150)
plt.close()


7. EVALUATION METRICS
Accuracy : 0.665
Precision: 0.663
Recall   : 0.670
F1-score : 0.667
ROC-AUC  : 0.739

Full classification report:
               precision    recall  f1-score   support

           0       0.67      0.66      0.66       100
           1       0.66      0.67      0.67       100

    accuracy                           0.67       200
   macro avg       0.67      0.67      0.66       200
weighted avg       0.67      0.67      0.66       200

Confusion Matrix:
 [[66 34]
 [33 67]]


FileNotFoundError: [Errno 2] No such file or directory: '/home/claude/project/confusion_matrix.png'

 # 8. COEFFICIENT INTERPRETATION

In [14]:
coefs = model.coef_[0]
coef_df = pd.DataFrame({'Feature': feature_cols, 'Coefficient': coefs})
coef_df['Odds_Ratio'] = np.exp(coef_df['Coefficient'])
coef_df = coef_df.sort_values('Coefficient', ascending=False)

print("\n" + "="*70)
print("8. FEATURE COEFFICIENTS (sorted, standardized features)")
print("="*70)
print(coef_df.to_string(index=False))
coef_df.to_csv('coefficients.csv', index=False)

plt.figure(figsize=(7,5))
sns.barplot(data=coef_df, x='Coefficient', y='Feature', hue='Feature', legend=False, palette='coolwarm')
plt.title('Logistic Regression Coefficients - Employee Attrition')
plt.tight_layout()
plt.savefig('coefficients_plot.png', dpi=150)
plt.close()

print("\nDone. Files saved: confusion_matrix.png, roc_curve.png, coefficients.csv, coefficients_plot.png")




8. FEATURE COEFFICIENTS (sorted, standardized features)
         Feature  Coefficient  Odds_Ratio
job_satisfaction     0.655048    1.925235
 promotion_years     0.613644    1.847149
  monthly_income     0.583814    1.792864
             age    -0.438686    0.644883
years_at_company    -0.446574    0.639817
  overtime_hours    -0.619121    0.538418

Done. Files saved: confusion_matrix.png, roc_curve.png, coefficients.csv, coefficients_plot.png


# 9. WRITTEN INTERPRETATION

In [15]:
tn, fp, fn, tp = cm.ravel()
print("\n" + "="*70)
print("9. CONFUSION MATRIX - INTERPRETATION")
print("="*70)
print(f"""
Out of {tn+fp+fn+tp} test employees:
  - {tn} who stayed were correctly predicted to stay (True Negatives)
  - {fp} who stayed were incorrectly flagged as at-risk (False Positives)
  - {fn} who left were missed by the model (False Negatives)
  - {tp} who left were correctly caught (True Positives)

Because the target is perfectly balanced (50/50), accuracy is a fair
headline metric here. Precision ({prec:.3f}) and recall ({rec:.3f}) are close
to each other and to accuracy -> the model is not biased toward either class.
ROC-AUC ({auc:.3f}) confirms decent (not outstanding) separation between
employees who leave and those who stay, across all thresholds.
""")

print("="*70)
print("10. COEFFICIENT INTERPRETATION (practical meaning)")
print("="*70)
print("""
Because features were standardized, coefficient magnitudes are directly
comparable:

  job_satisfaction (+): strongest POSITIVE predictor of leaving.
      -> COUNTERINTUITIVE: normally higher satisfaction should reduce
         attrition. Worth flagging to the reader rather than hiding it -
         may reflect a genuine pattern in this dataset (e.g. satisfied,
         high performers being poached externally), a confounding effect
         not captured by only 6 features, or an artifact of how the
         data was generated. Needs domain-expert review before acting on it.

  promotion_years (+): more years since last promotion -> higher odds of
      leaving. Matches typical HR intuition (stalled career -> attrition).

  monthly_income (+): higher income -> higher odds of leaving in this data.
      Also COUNTERINTUITIVE vs typical HR assumptions - same caveat as
      job_satisfaction above.

  overtime_hours (-): more overtime -> LOWER odds of leaving.
      Could reflect highly engaged/committed employees putting in extra
      hours, rather than overworked employees quitting.

  years_at_company (-): longer tenure -> lower odds of leaving.
      Matches typical HR intuition.

  age (-): older employees -> lower odds of leaving.
      Matches typical HR intuition.
""")

print("="*70)
print("11. LIMITATIONS, GENERALIZATION RISKS & IMPROVEMENTS")
print("="*70)
print("""
  - Two coefficients (job_satisfaction, monthly_income) contradict typical
    HR intuition. This should be investigated with domain expertise before
    the model is trusted operationally - it may indicate omitted variables,
    a genuine but surprising pattern, or noise in only 6 available features.
  - Only 6 features are available (no department, role, manager, engagement
    survey text) - likely limits how much signal the model can extract,
    consistent with a moderate (not strong) AUC of 0.739.
  - Logistic Regression assumes a LINEAR log-odds relationship between each
    feature and the outcome. True relationships (e.g. a U-shaped tenure
    effect) would not be captured without adding interaction or polynomial
    terms - especially worth testing given the counterintuitive coefficients.
  - Tree-based models (Random Forest, Gradient Boosting) could be tried as
    a comparison since they capture non-linear/interaction effects that
    logistic regression cannot.
  - The classification threshold (default 0.5) should be tuned based on the
    real business cost of missing an at-risk employee vs. the cost of an
    unnecessary retention intervention.
""")

print("="*70)
print("12. CONCLUSION")
print("="*70)
print(f"""
Logistic Regression achieves {acc:.1%} accuracy and an AUC of {auc:.3f} on this
balanced attrition dataset - a moderate improvement over the 50% a random
guess would achieve. The coefficient analysis surfaces both intuitive
relationships (years_at_company and age reducing attrition odds,
promotion_years increasing them) and some that contradict typical HR
assumptions (job_satisfaction and monthly_income both increasing odds of
leaving), which should be investigated further with domain expertise rather
than acted on directly. Overall this is a reasonable, interpretable
baseline, but the counterintuitive signals suggest richer features or a
non-linear model would strengthen both accuracy and practical trust in
the conclusions.
""")



9. CONFUSION MATRIX - INTERPRETATION

Out of 200 test employees:
  - 66 who stayed were correctly predicted to stay (True Negatives)
  - 34 who stayed were incorrectly flagged as at-risk (False Positives)
  - 33 who left were missed by the model (False Negatives)
  - 67 who left were correctly caught (True Positives)

Because the target is perfectly balanced (50/50), accuracy is a fair
headline metric here. Precision (0.663) and recall (0.670) are close
to each other and to accuracy -> the model is not biased toward either class.
ROC-AUC (0.739) confirms decent (not outstanding) separation between
employees who leave and those who stay, across all thresholds.

10. COEFFICIENT INTERPRETATION (practical meaning)

Because features were standardized, coefficient magnitudes are directly
comparable:

  job_satisfaction (+): strongest POSITIVE predictor of leaving.
      -> COUNTERINTUITIVE: normally higher satisfaction should reduce
         attrition. Worth flagging to the reader rather th